In [1]:
import re
import glob
import zipfile
import time
from pathlib import Path
import json
import requests
import psutil
import pandas as pd
from urllib.request import urlretrieve


In [2]:
# ------Flags
FORCE_DOWNLOAD = False # for downloading zip from api request
FORCE_MERGE = False # for combining CSV via pandas

In [3]:
# # Same decorator as in Milestone 1, extended with peak-measurement support.
# # If the decorated function uses `yield`, the decorator snapshots RSS at the
# # yield point (while the big allocation is still alive) then lets the function
# # finish.  Normal (non-generator) functions are measured after return as before.
# _proc = psutil.Process(os.getpid())


# # TODO add this into a utils folder and just import it to be cleaner 

# def measure(fn):
#     _is_gen = inspect.isgeneratorfunction(fn)

#     @functools.wraps(fn)
#     def wrapper(*args, **kwargs):
#         gc.collect()
#         m0 = _proc.memory_info().rss / 1e6
#         c0 = time.process_time()
#         t0 = time.perf_counter()
#         if _is_gen:
#             gen = fn(*args, **kwargs)
#             yielded = next(gen)                    # run to yield: allocation is live
#             if yielded is not None:
#                 wrapper.mem_mb = yielded           # function passed its own measurement
#             else:
#                 wrapper.mem_mb = _proc.memory_info().rss / 1e6 - m0
#             try:
#                 next(gen)                          # run to return
#             except StopIteration as exc:
#                 out = exc.value
#         else:
#             out = fn(*args, **kwargs)
#             wrapper.mem_mb = _proc.memory_info().rss / 1e6 - m0
#         wrapper.wall = time.perf_counter() - t0
#         wrapper.cpu  = time.process_time() - c0
#         print(
#             f"{fn.__name__}:  wall={wrapper.wall:.1f}s  "
#             f"cpu={wrapper.cpu:.1f}s  "
#             f"mem \u0394{wrapper.mem_mb:+.0f} MB"
#         )
#         return out
#     return wrapper

In [4]:
# ------Local Directory Download Set up 
WORKDIR = Path.cwd() # current notebook directory
OUTPUT_DATA_DIR = WORKDIR / "../data" 
OUTPUT_DATA_DIR.mkdir(exist_ok=True) # ignores/suppress error is dir already exists

OUTPUT_ZIP_DIR = OUTPUT_DATA_DIR / "zip_downloads"
OUTPUT_EXTRACT_DIR = OUTPUT_DATA_DIR / "extracted_downloads"
OUTPUT_COMBINED_DIR =  OUTPUT_DATA_DIR / "clean"

# ------API Set up 
article_id = 14096681  # unique identifier of the article/dataset on figshare
url = f"https://api.figshare.com/v2/articles/{article_id}" # full 'url' for API
headers = {"Content-Type": "application/json"}

In [5]:
# ------API GET request 
response = requests.request("GET", url, headers=headers)
api_data = json.loads(response.text)  
available_files = api_data["files"] # View available data files/downloads
target_data_file = ["data.zip"]  # "name" of data download want

available_files

[{'id': 26579150,
  'name': 'daily_rainfall_2014.png',
  'size': 58863,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579150',
  'supplied_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'computed_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'mimetype': 'image/png'},
 {'id': 26579171,
  'name': 'environment.yml',
  'size': 192,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579171',
  'supplied_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'computed_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'mimetype': 'text/plain'},
 {'id': 26586554,
  'name': 'README.md',
  'size': 5422,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26586554',
  'supplied_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'computed_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'mimetype': 'text/x-python'},
 {'id': 26766812,
  'name': 'data.zip',
  'size': 814041183,
  'is_link_only': False,
  'download_url': 'https://n

In [6]:
# ------Download Zip(s)
for file in available_files:
    if file["name"] in target_data_file:

        output_zip_path = OUTPUT_ZIP_DIR / file["name"]

        if output_zip_path.exists() and not FORCE_DOWNLOAD: # If download already occurred, does nothing
            print(f"{file['name']} already exists. Skipping.")
        else:
            output_zip_path.parent.mkdir(exist_ok=True)  # ensure folder exists

            print(f"Downloading {file['name']}...")
            urlretrieve(file["download_url"], output_zip_path)
            print(f"Downloaded {file['name']} without errors")


data.zip already exists. Skipping.


In [7]:
# ------Extract files in Zip
zip_path =  OUTPUT_ZIP_DIR / "data.zip" # TODO (not to hardcode)

if not OUTPUT_EXTRACT_DIR.exists():
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(OUTPUT_EXTRACT_DIR)
        print("Files now exported.")
else:
    print("Files already extracted. Skipping extraction.")

Files already extracted. Skipping extraction.


In [10]:
# ------Combine CSV files via Pandas

def combine_csvs(input_dir,
                output_dir,
                columns, 
                skip_files = None,
                force_merge = False):
    if output_dir.exists() and not force_merge:
        print("Merged file already exists. Skipping.")
    else: 
        dataframes = [] # list of all the subdf's to combine

        files = glob.glob(str(input_dir / "*.csv")) # "For all files in extracts folder:"
        files_len = len(files)

        for count, file in enumerate(files, start=1):
            filename = Path(file).name # "what is the file name?"
            print(f"Progress: ({count}/{files_len}), processing: {filename}        ",
                    end="\r", flush=True) # Add print feedback of progress

            if filename in skip_files: # "does the file need to be skipped?"
                print(f"Skipped {filename}") 
                continue

            sub_df = pd.read_csv(file,  # "read the csv as a df..."
                                index_col=0, 
                                usecols=columns) 
            sub_df["model"] = re.search(r"^([^_]+)", filename).group(1) # add model (filename) as a column

            dataframes.append(sub_df) # add to list to be combined

        combined_df = pd.concat(dataframes) # after all subdf created, concat them into a combined df
        combined_df.to_csv(output_dir/"combined.csv") # export as a csv 
        print(f"Combined {len(dataframes)} files into {output_dir}")

use_cols = ["time","lat_min","lat_max","lon_min","lon_max","rain (mm/day)"]
SKIP_FILES = ["observed_daily_rainfall_SYD.csv"]

combine_csvs(input_dir = OUTPUT_EXTRACT_DIR,
            output_dir = OUTPUT_COMBINED_DIR,
            columns =  use_cols,
            skip_files = SKIP_FILES,
            force_merge = FORCE_MERGE)

KeyboardInterrupt: 

Record results in the table below.
Based on the comparison, in 3–5 sentences, reflect on what you observe: how do the numbers differ across machines? What does the wall/cpu ratio tell you about where the bottleneck is? What surprised you?

Warning: Some of you might not be able to do it on your laptop. It's fine if you're unable to do it. Just make sure you discuss the reasons why you might not have been able to run this on your laptop.